In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:

prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [ ]:

eurusd = pd.read_csv("common/MachineLearningModel/output/one_mins/EURUSD_1_Min.csv")
eurjpy = pd.read_csv('common/MachineLearningModel/output/one_mins/EURJPY_1_Min.csv')
eurcad = pd.read_csv('common/MachineLearningModel/output/one_mins/EURCAD_1_Min.csv')
euraud = pd.read_csv('common/MachineLearningModel/output/one_mins/EURAUD_1_Min.csv')
eurgbp = pd.read_csv('common/MachineLearningModel/output/one_mins/EURGBP_1_Min.csv')

In [ ]:
from ta.trend import macd,cci,adx,macd_signal,adx_pos,adx_neg
from ta.momentum import rsi,stochrsi_d,stochrsi_k,stochrsi
def calculate(pd: pd.DataFrame):
    pdrsi = rsi(pd['close'],14)
    # rsi.dropna(axis=0,inplace=True)
    pdcci = cci(pd['high'],pd['low'],pd['close'],14)
    # cci.dropna(axis=0,inplace=True)
    pdadx = adx(pd['high'],pd['low'],pd['close'])
    pdadx_pos = adx_pos(pd['high'],pd['low'],pd['low']) 
    pdadx_neg = adx_neg(pd['high'],pd['low'],pd['low'])
    # adx.dropna(axis=0,inplace=True)
    pdmacd = macd(pd['close'])
    # macd.dropna(axis=0,inplace=True)
    pdmacd_signal = macd_signal(pd['close'])
    # macd_signal.dropna(axis=0,inplace=True)
    pdstochrsi_d = stochrsi_d(pd['close'])
    pdstochrsi_k = stochrsi_k(pd['close'])
    pdstochrsi = stochrsi(pd['close'])
    # stochrsi.dropna(axis=0,inplace=True)
    pd2 = pd.iloc[:,1:7].copy(deep=True) # iloc[row,column]
    pd2['rsi'] = pdrsi
    pd2['cci'] = pdcci
    pd2['adx'] = pdadx
    pd2['adx_pos'] = pdadx_pos
    pd2['adx_neg'] = pdadx_neg
    pd2['macd'] = pdmacd
    pd2['macd_signal'] = pdmacd_signal
    pd2['stochrsi_d'] = pdstochrsi_d
    pd2['stochrsi_k'] = pdstochrsi_k
    pd2['stochrsi'] = pdstochrsi
    return pd2


In [ ]:
pd1 = calculate(eurusd)
pd2 = calculate(eurjpy)
pd3 = calculate(eurcad)
pd4 = calculate(euraud)
pd5 = calculate(eurgbp)

In [ ]:
data = pd.concat([pd1,pd2,pd3,pd4,pd5])
data.dropna(axis=0,inplace=True)
# print(data.columns)
# print(data.count())
print(data.tail(-1))

In [ ]:
data['RSI_1'] = np.where(data['rsi'] < 30, 1, np.where(data['rsi'] > 70, 2, 0))
# data['MACD_1'] = np.where(data['macd'] < data['macd_signal'], 2, np.where(data['macd'] > data['macd_signal'], 1, 0))
# data['CCI_1'] = np.where(data['cci'] < -80, 1, np.where(data['cci'] > 80, 2, 0))
adx_condition = (data['adx'] > 25.00) | (data['rsi'] > 70)
adx_condition_2 = (data['adx'] > 25.00) | (data['rsi'] < 30)
data['ADX_1'] = np.where((data['adx'] > 25.00) & (data['adx_pos'] < data['adx_neg']), 1, np.where((data['adx'] > 25.00) & (data['adx_pos'] > data['adx_neg']), 2, 0))
conditions_3 = (data['stochrsi'] > 0.75) & (data['stochrsi_k'] < data['stochrsi_d'])
conditions_4 = (data['stochrsi'] < 0.25) & (data['stochrsi_k'] > data['stochrsi_d'])
data['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0))
conditions_2 = (data['ADX_1'] == 2)
conditions_1 = (data['ADX_1'] == 1)
data['Prediction'] = np.where(adx_condition_2 & (data['open'] < data['close']), 1,
                              np.where(adx_condition & (data['open'] > data['close']), 2, 0))


In [ ]:
# data['rsi'] = data['rsi'].astype(dtype=int)
# data['cci'] = data['cci'].astype(dtype=int)
# data['adx'] = data['adx'].astype(dtype=int)
# data['adx_pos'] = data['adx_pos'].astype(dtype=int)
# data['adx_neg'] = data['adx_neg'].astype(dtype=int)

In [ ]:
# data.drop(axis=1,labels=['rsi','cci','adx','macd','macd_signal','stochrsi','stochrsi_k','stochrsi_d'],inplace=True)
# data.drop(axis=1,labels=['RSI_1'],inplace=True)

In [ ]:
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.head())
print(data.shape)


In [ ]:


le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
# Generate a synthetic binary classification dataset
X = data.iloc[:,6:-1]
y = data.iloc[:, -1]

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Define the base learners
base_learners = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
    ('xgb', XGBClassifier(booster="gbtree",max_depth=9,min_child_weight = 2))
]
# Define the meta-learner
meta_learner = LogisticRegression()
# Build the Stacking classifier
stacking_clf = StackingClassifier(estimators=base_learners, final_estimator=meta_learner)
# Train the Stacking classifier
stacking_clf.fit(X_train, y_train)
# Evaluate the model
stacking_clf.score(X_test, y_test)

In [ ]:
# import pickle
# combine_final_model = pickle.dump(stacking_clf, open('combineclassifier.sav','wb'))

In [ ]:
print(data['Prediction'].value_counts())
X = data.iloc[:,6:-1]
y = data.iloc[:, -1]
print(data.columns)
print(X.head())
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.2, random_state = 24)



In [ ]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=9,min_child_weight = 2)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


In [ ]:
final_xgb_model = XGBClassifier()
final_xgb_model.fit(X, y)

In [ ]:
import pickle
# xgb_final_model = pickle.dump(final_xgb_model, open('xgbclassifier.sav','wb'))

In [ ]:
from TradingDataGenerate import main
s = main.TvDatafeed('mageshragav1@gmail.com','Magesh1@')


In [ ]:
import random
# symbols = random.choice(['EURUSD','EURJPY','GBPUSD','EURGBP'])
symbols = 'EURJPY'
print(symbols)
response_data = s.get_hist(symbol=symbols,exchange='FX',interval=main.Interval.in_15_minute,n_bars=150,extended_session=False)
pd3 = calculate(response_data)
print(pd3.iloc[-1])
pd3['RSI_1'] = np.where(pd3['rsi'] < 30, 1, np.where(pd3['rsi'] > 70, 2, 0))
pd3['ADX_1'] = np.where((pd3['adx'] > 25.00) & (pd3['adx_pos'] < pd3['adx_neg']), 1, np.where((pd3['adx'] > 25.00) & (pd3['adx_pos'] > pd3['adx_neg']), 2, 0))
conditions_3 = (pd3['stochrsi'] > 0.75) & (pd3['stochrsi_k'] < pd3['stochrsi_d'])
conditions_4 = (pd3['stochrsi'] < 0.25) & (pd3['stochrsi_k'] > pd3['stochrsi_d'])
pd3['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0))
# pd3['RSI_1'] = np.where(pd3['rsi'] < 30, 1, np.where(pd3['rsi'] > 70, 2, 0))
pd3.dropna(inplace=True)
pd3.reset_index()
print(pd3.iloc[-1])
data_1 = pd3.iloc[-1,5:].to_dict()
data_1 = pd.DataFrame({key: [value] for key, value in data_1.items()})
print(data_1)
output = final_xgb_model.predict(pd.DataFrame(data_1))
print(output)